# CANguard -- EDA (thin orchestrator)

This notebook orchestrates `canguard.data` only. No parsing logic lives here;
loader behaviour is tested in the `canguard` test suite.


## 1. Load (orchestrated)


In [ ]:
from pathlib import Path

from canguard.data import get_loader

DATA_DIR = Path('HCRL Car-Hacking')
SAMPLE_SIZE = 60000

# Load all four datasets with the registered factory.
samples = {
    name: get_loader('hcrl', DATA_DIR / f'{name}_dataset.csv').load(sample_size=SAMPLE_SIZE)
    for name in ['DoS', 'Fuzzy', 'RPM', 'gear']
}
for name, df in samples.items():
    pct = df['is_attack'].mean() * 100
    print(f'{name:>6s}: {len(df):>7,} rows  ({pct:.2f}% attack, {df["can_id"].nunique():,} IDs)')


## 2. Class balance


In [ ]:
for name, df in samples.items():
    counts = df['label'].value_counts().to_dict()
    print(f'{name:>6s}: {counts}')


## 3. Naive-rule baseline (dataset limitation check)


In [ ]:
# All HCRL attacks are presence-based: new/dominant IDs or frozen bytes.
print('See tests/test_hcrl_loader.py for the DLC-aware schema equivalence checks.')
print('See eda_hcrl.ipynb (git history) for the full exploratory analysis.')
for name, df in samples.items():
    normal_ids = set(df.loc[df['label'] == 'R', 'can_id'].unique())
    att_ids = set(df.loc[df['is_attack'] == 1, 'can_id'].unique())
    new_ids = att_ids - normal_ids
    dom = df.loc[df['is_attack'] == 1, 'can_id'].value_counts().head(1)
    print(f'{name:>6s}: new attack ids={len(new_ids):>5,}  dominant attack id={list(dom.index)[0] if len(dom) else "-"}  ({dom.iloc[0] if len(dom) else 0:,} msgs)')
